# Advice API Tool
`adviceApiTool.py` is the MAF implementation of the health.gov API tool.
It extends `BaseAPITool` from `baseApiTool.py`.
To see the parent class, check `baseApiTool.ipynb`.

## AG2 → MAF Migration
| AG2 Pattern | MAF Equivalent |
|---|---|
| `AdviceAPIToolInput(BaseAPIToolInput)` Pydantic model | Typed parameters on `__call__` |
| `Field(description=...)` per parameter | Inline docstring `Args:` block |
| `args_schema: Type[BaseModel] = AdviceAPIToolInput` | Removed — MAF reads `__call__` annotations |
| `apiTool._run(**query.model_dump())` in example | `apiTool(age="35", sex="female", ...)` directly |

The `parse_response` override, the health.gov API URL, and all parameter names are unchanged.
The only thing that changed is how the tool presents itself to the framework.

## Imports
Since this notebook lives in `docs/tools/`, we adjust `sys.path` to resolve imports from the project root.


In [ ]:
import sys
from pathlib import Path

def setup_path():
    ai_root = Path().resolve().parent.parent
    if str(ai_root) not in sys.path:
        sys.path.insert(0, str(ai_root))

setup_path()

In [ ]:
from typing import Optional
from agent_framework import tool
from tools.baseApiTool import BaseAPITool
import requests

## AdviceAPITool Class
In AG2, the health.gov-specific parameters were defined on `AdviceAPIToolInput(BaseAPIToolInput)` —
a Pydantic model that AG2 serialized into a JSON function schema via `generate_llm_config(tool)`.

The parameter descriptions were critical because they were the LLM's only instructions about valid values.
For example, `age` explicitly says `(Enter as a string)` — without that, the model sends an integer
and the health.gov API rejects the request.

In MAF, those descriptions move into the `__call__` docstring `Args:` block.
MAF reads them the same way it read `Field(description=...)` — they become the parameter descriptions
in the tool schema sent to the LLM. The same precision is required, just in a different place.

`parse_response` is unchanged — it still trims the health.gov JSON payload
down to the first section of the first resource to keep the context window reasonable.


In [ ]:
class AdviceAPITool(BaseAPITool):

    base_url: str = "https://odphp.health.gov/myhealthfinder/api/v4/myhealthfinder.json"

    def parse_response(self, response: requests.Response):
        data = response.json()
        try:
            return data["Result"]["Resources"]["All"]["Resource"][0]["Sections"]["section"][0]
        except Exception:
            return data

    @tool
    def __call__(
        self,
        age: Optional[str] = None,
        sex: Optional[str] = None,
        pregnant: Optional[str] = None,
        sexually_active: Optional[str] = None,
        tobacco_use: Optional[str] = None,
    ) -> dict:
        """
        Query the health.gov API for personalized healthcare advice.
        All parameters are optional — provide whichever are known.

        Args:
            age: Age of the person. Enter as a string (e.g. "35"), not an integer.
            sex: Sex of the person. Use "male" or "female".
            pregnant: Pregnancy status. Use "yes" or "no".
            sexually_active: Sexually active status. Use "yes" or "no".
            tobacco_use: Tobacco use status. Use "yes" or "no".
        """
        # health.gov expects camelCase query params — map from snake_case
        params = {k: v for k, v in {
            "age": age,
            "sex": sex,
            "pregnant": pregnant,
            "sexuallyActive": sexually_active,
            "tobaccoUse": tobacco_use,
        }.items() if v is not None}

        import requests as _requests
        try:
            response = _requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            return self.parse_response(response)
        except _requests.exceptions.RequestException as e:
            return {"Error": str(e)}

## Example Run
The health.gov API is free and unlimited.

In AG2, the example called `apiTool._run(**query.model_dump())` — you had to instantiate the
Pydantic input model first and then unpack it. That was necessary because `_run` was the LangChain
internal entry point, not a normal callable.

In MAF, the tool is just called directly with keyword arguments.


In [ ]:
api_tool = AdviceAPITool()

result = api_tool(
    age="35",
    sex="female",
    pregnant="no",
    sexually_active="yes",
    tobacco_use="no",
)

print(result)